# 🤝 W10: 협력사 커뮤니케이션 자동화
**hexa-1 — Week 10** | ⏱️ 60분 | 🎯 협력사별 맞춤 메일 + 발주서 자동화

> 납품 지연·불량 발생 시 협력사에게 보내는 공문/메일을 AI가 자동 초안 작성

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q pandas matplotlib google-generativeai requests
import pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
nanum=next((f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name),None)
if nanum: plt.rcParams['font.family']=fm.FontProperties(fname=nanum).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash')
print('✅ 환경 설정 완료')


## Step 1: 협력사 및 이슈 정보 입력 (✏️)

In [ ]:
COMPANY = {'name': '✏️ [우리 회사명]', 'representative': '✏️ [담당자명]', 'dept': '구매팀'}
VENDORS = [
    {'name': '✏️ [협력사 A]', 'issue': '납품 지연 3일', 'type': 'delay', 'part': '브래킷 A형'},
    {'name': '✏️ [협력사 B]', 'issue': '불량률 4.2% 초과', 'type': 'defect', 'part': '샤프트 C형'},
    {'name': '✏️ [협력사 C]', 'issue': '3월 발주 확인 요청', 'type': 'order', 'part': '볼트 세트'},
]
print(f'✅ 협력사 {len(VENDORS)}개 처리 예정')


## Step 2: 유형별 공문 자동 생성

In [ ]:
TEMPLATES = {
    'delay': '납품 지연에 대한 원인 및 향후 일정을 명시하는 공식 개선 요청 공문',
    'defect': '불량률 초과에 대한 원인 분석 및 재발 방지 대책 요청 공문',
    'order': '다음 달 발주 수량 및 일정 확인 요청 공문',
}
letters = {}
for v in VENDORS:
    template_desc = TEMPLATES.get(v['type'], '일반 협력사 커뮤니케이션 공문')
    prompt = f"""제조업 구매팀 담당자로서 협력사에 보낼 공문을 작성해주세요.
발신: {COMPANY['name']} {COMPANY['dept']} {COMPANY['representative']}
수신: {v['name']}
부품명: {v['part']}
이슈: {v['issue']}
문서 유형: {template_desc}
형식: 마크다운, 공식적 문체, 수신 확인란 포함"""
    resp = model.generate_content(prompt)
    letters[v['name']] = resp.text
    print(f'✅ {v["name"]} 공문 생성 완료')


## Step 3: 일괄 저장 & 다운로드

In [ ]:
import os, zipfile
os.makedirs('vendor_letters',exist_ok=True)
for name,content in letters.items():
    fname=f'vendor_letters/{name.replace(" ","_")}.md'
    with open(fname,'w',encoding='utf-8') as f: f.write(content)
with zipfile.ZipFile('vendor_letters.zip','w') as zf:
    for n in letters: zf.write(f'vendor_letters/{n.replace(" ","_")}.md')
from google.colab import files
files.download('vendor_letters.zip')
print(f'✅ {len(letters)}건 협력사 공문 다운로드 완료!')


---
## 🔥 확장 과제
1. 이메일 자동 발송(SMTP) 연결
2. W9 이상감지 → 자동 협력사 알림 파이프라인
3. 언어 선택: 한국어/영어 공문 동시 생성